In [ ]:
import os
from pathlib import Path

# Set writable cache locations before importing Spateo/Numba-backed modules.
os.environ.setdefault("XDG_CACHE_HOME", "/private/tmp/cache-spateo")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-spateo")
os.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/numba-spateo")

import anndata as ad
import numpy as np
import pandas as pd
import spateo
from spateo.tools.glm import glm_degs


In [ ]:

data_dir = Path("../data")
counts_path = "../data/ca3_simulation_gradient/ca3_simulated_dataset_peakt_1_sd.r_2_rep_01.csv"
digitized_path = data_dir / "ca3_xy_digitized.csv"

counts_df = pd.read_csv(counts_path, index_col=0)
digitized_coords = pd.read_csv(digitized_path, index_col=0)

required_digitized_cols = ["xcoord", "ycoord", "spateo_longitudinal", "spateo_radial"]
missing_cols = [col for col in required_digitized_cols if col not in digitized_coords.columns]
if missing_cols:
    raise ValueError(f"Missing columns in {digitized_path}: {missing_cols}")

display(digitized_coords[required_digitized_cols].head())

# The simulated counts CSV is genes x spots; AnnData expects spots x genes.
raw_counts = counts_df.to_numpy(dtype=float).T
gene_names = counts_df.index.astype(str).to_list()

if raw_counts.shape[0] != digitized_coords.shape[0]:
    raise ValueError(
        f"Mismatched spots: counts={raw_counts.shape[0]}, digitized_coords={digitized_coords.shape[0]}"
    )

# Spateo's glm_degs expects size-factor-normalized expression.
library_size = raw_counts.sum(axis=1)
median_library_size = np.median(library_size[library_size > 0])
size_factors = np.where(library_size > 0, library_size / median_library_size, 1.0)
normalized_counts = raw_counts / size_factors[:, None]

obs = pd.DataFrame(
    {
        "xcoord": digitized_coords["xcoord"].to_numpy(),
        "ycoord": digitized_coords["ycoord"].to_numpy(),
        "t": digitized_coords["spateo_longitudinal"].to_numpy(),
        "r": digitized_coords["spateo_radial"].to_numpy(),
        "size_factor": size_factors,
        "library_size": library_size,
    },
    index=[f"spot_{i + 1}" for i in range(raw_counts.shape[0])],
)

adata = ad.AnnData(
    X=normalized_counts,
    obs=obs,
    var=pd.DataFrame(index=gene_names),
)
adata.layers["counts"] = raw_counts
adata.obsm["spatial"] = digitized_coords[["xcoord", "ycoord"]].to_numpy()
adata.var_names_make_unique()

print(adata)
print(f"t range: {adata.obs['t'].min():.4f} to {adata.obs['t'].max():.4f}")
print(f"r range: {adata.obs['r'].min():.4f} to {adata.obs['r'].max():.4f}")


In [ ]:
#Plot t and r to verify they look correct
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc1 = axes[0].scatter(adata.obs["xcoord"], adata.obs["ycoord"], c=adata.obs["r"], cmap="viridis")
axes[0].set
axes[0].set_title("t (longitudinal)")
plt.colorbar(sc1, ax=axes[0])
plt.show()

sc2 = axes[1].scatter(adata.obs["xcoord"], adata.obs["ycoord"], c=adata.obs["r"], cmap="viridis")
axes[1].set_title("r (radial)")
axes[1].set
axes[1].set_title("r (radial)")
plt.colorbar(sc2, ax=axes[1])
plt.show()

In [ ]:
glm_key = "glm_degs_t_linear"

import warnings

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="Negative binomial dispersion parameter alpha not set.*"
    )

    glm_degs(
        adata,
        key_added=glm_key,
        fullModelFormulaStr="~cr(t, df=9) + cr(r, df=5)",
        reducedModelFormulaStr = "~cr(r,df=5)",
        qval_threshold=None,
        llf_threshold=None,
        inplace=True,
    )

glm_hits = adata.uns[glm_key]["glm_result"].copy()
glm_hits.insert(0, "gene", glm_hits.index)
glm_hits = glm_hits.sort_values(["qval", "pval", "log-likelihood"])

print("Top spatially variable genes from Spateo GLM using full model ~t vs reduced model ~1:")
display(glm_hits.head(25))

print("Top hits as text:")
print(glm_hits.head(25).to_string(index=False))

#Find the rank of the gene 1
gene_1_rank = glm_hits.index.get_loc("1") + 1
print(f"Rank of gene_1: {gene_1_rank}")
